In [ ]:
!pip install qsimov

In [ ]:
import numpy as np
import qsimov as qs
import math
import unittest

In [ ]:
def to_bin(a,n):
    exponent = np.arange(0,n,1)
    result = 2 ** exponent
    temp = a // result
    positions = temp & 1
    return positions

In [ ]:
def get_rs():
    gate = qs.QGate(3, 0, "RS")
    gate.add_operation("X", targets=2, controls=0)
    gate.add_operation("X", targets=2, controls=1)
    return gate

In [ ]:
def get_ra(n):
    num_qubits = 3*n+1
    gate = qs.QGate(num_qubits, 0, "RA") # Reverse Adder
    carry = get_carry()
    rs = get_rs()
    for i in range(n-1):
        temp = 3*i
        gate.add_operation(rs,targets=[temp,temp+1,temp+2])
        gate.add_operation(carry, targets=[temp,temp+1,temp+2,temp+3])
    temp += 3
    gate.add_operation(rs,targets=[temp,temp+1,temp+2])
    gate.add_operation("X", targets=temp+2, controls=temp+1)
    rc = get_rc() # get reverse carry
    gate.add_operation(rc,targets=[temp,temp+1,temp+2,temp+3])
    for i in range(n-2,-1,-1):
        temp = 3*i
        gate.add_operation(rc,[temp,temp+1,temp+2,temp+3])
    return gate

In [ ]:
def invmodn(b,n):

#function y = invmodn( b,n);
    '''
    This function calculates the inverse of an element b mod n
    It uses the extended euclidean algorithm
    '''

    n0=n
    b0=b
    t0=0
    t=1

    q=math.floor(n0/b0)
    r=n0-q*b0

    while r>0:
        temp=t0-q*t
        if temp>=0:
            #(temp >=0),
            #temp=mod(temp,n);
            temp = temp % n
        #end;
        if temp < 0:
            temp= n - ( (-temp) % n) #mod(-temp,n));
        #end;
        t0=t
        t=temp
        n0=b0
        b0=r
        q=math.floor(n0/b0)
        r=n0-q*b0
    #end

    if b0 !=1:
        y=[]

        #disp('No inverse');
    else:
        y = t % n
        #return y
        #y=mod(t,n)
    #end;
    return y

In [ ]:
def get_sum():
    gate = qs.QGate(3, 0, "SUM")
    gate.add_operation("X", targets=2, controls=1)
    gate.add_operation("X", targets=2, controls=0)
    return gate

In [ ]:
def get_carry():
    gate = qs.QGate(4, 0, "CARRY")
    gate.add_operation("X", targets=3, controls=[1,2])
    gate.add_operation("X", targets=2, controls=1)
    gate.add_operation("X", targets=3, controls=[0,2])
    return gate

In [ ]:
def get_rc():
    gate = qs.QGate(4, 0, "RC")
    gate.add_operation("X", targets=3, controls=[0,2])
    gate.add_operation("X", targets=2, controls=1)
    gate.add_operation("X", targets=3, controls=[1,2])
    return gate

In [ ]:
def get_adder(n):
    num_qubits = 3*n+1
    gate = qs.QGate(num_qubits, 0, "ADDER")
    carry = get_carry()
    for i in range(n-1):
        temp = 3*i
        gate.add_operation(carry, targets=[temp,temp+1,temp+2,temp+3])
    temp += 3
    gate.add_operation(carry,targets=[temp,temp+1,temp+2,temp+3])
    gate.add_operation("X", targets=temp+2, controls=temp+1)
    sum = get_sum()
    gate.add_operation(sum,targets=[temp,temp+1,temp+2])
    rc = get_rc() # get reverse carry
    for i in range(n-2,-1,-1):
        temp = 3*i
        gate.add_operation(rc,[temp,temp+1,temp+2,temp+3])
        gate.add_operation(sum, targets=[temp,temp+1,temp+2])
    return gate

In [ ]:
def get_modadd(N,n):
    num_qubits = 4*n+2
    gate = qs.QGate(num_qubits, 0, "MODADDER") # Modular Adder

    # first adder
    adder = get_adder(n)
    num_qubits_adder = 3*n+1
    targets = np.arange(0,num_qubits_adder,1)
    gate.add_operation(adder,targets=targets)

    #swaps
    positions = np.arange(0,n,1) + num_qubits_adder
    for i in range(n):
        from_ = i*3+1
        to = positions[i]
        gate.add_operation("SWAP", targets=[from_,to])

    # reverse adder
    ra = get_ra(n)
    gate.add_operation(ra,targets=targets)

    #anticontrol
    target = num_qubits-1
    anticontrol = num_qubits_adder-1
    gate.add_operation("X", targets=target, anticontrols=anticontrol)

    #cancel N
    control = target
    exponent = np.arange(0,n,1)
    indices = exponent*3+1
    positions = to_bin(N,n)
    positions = positions == 1
    targets = indices[positions]
    for i in targets:
        gate.add_operation("X", targets=i, controls=control)


    # second adder
    targets_ = np.arange(0,num_qubits_adder,1)
    gate.add_operation(adder,targets=targets_)


    #reassign N
    for i in targets:
        gate.add_operation("X", targets=i, controls=control)



    #swaps
    positions = np.arange(0,n,1) + num_qubits_adder
    for i in range(n):
        from_ = i*3+1
        to = positions[i]
        gate.add_operation("SWAP", targets=[from_,to])



    # reverse adder
    gate.add_operation(ra,targets=targets_)


    #control
    target = num_qubits-1
    control = num_qubits_adder-1
    gate.add_operation("X", targets=target, controls=control)


    #third adder
    gate.add_operation(adder,targets=targets_)

    return gate

In [ ]:
def get_mod_mult(a,N,n):
    num_qubits = 5*n+3
    gate = qs.QGate(num_qubits, 0, "MODMULT") # Modular Multiplier
    num_qubits_mod_add = 4*n+2
    targets_mod_add = np.arange(0,num_qubits_mod_add,1) + n + 1
    exponent = np.arange(0,n,1)
    result = 2 ** exponent
    temp = a * result
    temp = temp % N
    indices = np.arange(0,n,1)
    indices = indices*3+1
    indices = indices + n + 1
    for i in range(n):
        temp2 = temp[i] // result
        positions = temp2 & 1
        positions = positions == 1
        if (np.sum(positions)!=0):
            targets = indices[positions]
            gate.add_operation("X", targets=targets, controls=[0,i+1])
            modadd = get_modadd(N,n)
            gate.add_operation(modadd, targets=targets_mod_add)
            gate.add_operation("X", targets=targets, controls=[0,i+1])
    for i in range(n):
        gate.add_operation("X", targets=i*3+2+n+1, anticontrols=0,controls=i+1)
    return gate

In [ ]:
def get_rev_modadd(N,n):
    num_qubits = 4*n+2
    gate = qs.QGate(num_qubits, 0, "RMA") # Reverse Modular Adder

    # reverse adder
    ra = get_ra(n)
    num_qubits_adder = 3*n+1
    targets = np.arange(0,num_qubits_adder,1)
    gate.add_operation(ra,targets=targets)

    # control
    target = num_qubits-1
    control = num_qubits_adder-1
    gate.add_operation("X", targets=target, controls=control)


    # adder
    adder = get_adder(n)
    gate.add_operation(adder,targets=targets)


    #swaps
    positions = np.arange(0,n,1) + num_qubits_adder
    for i in range(n-1,-1,-1):
        from_ = i*3+1
        to = positions[i]
        gate.add_operation("SWAP", targets=[from_,to])


    #reassign N
    control = target
    exponent = np.arange(0,n,1)
    indices = exponent*3+1
    positions = to_bin(N,n)
    positions = positions == 1
    targets = indices[positions]
    for i in targets:
        gate.add_operation("X", targets=i, controls=control)


    # reverse adder
    targets_ = np.arange(0,num_qubits_adder,1)
    gate.add_operation(ra,targets=targets_)


    #cancel N
    for i in targets:
        gate.add_operation("X", targets=i, controls=control)


    #anticontrol
    anticontrol = num_qubits_adder-1
    target = num_qubits-1
    gate.add_operation("X", targets=target, anticontrols=anticontrol)


    # adder
    gate.add_operation(adder,targets=targets_)


    #swaps
    positions = np.arange(0,n,1) + num_qubits_adder
    for i in range(n-1,-1,-1):
        from_ = i*3+1
        to = positions[i]
        gate.add_operation("SWAP", targets=[from_,to])

    # reverse adder
    gate.add_operation(ra,targets=targets_)

    return gate

In [ ]:
def get_rev_mod_mult(a,N,n):
    num_qubits = 5*n+3
    gate = qs.QGate(num_qubits, 0, "RMM") # Reverse Modular Multiplier

    for i in range(n-1,-1,-1):
        gate.add_operation("X", targets=i*3+2+n+1, anticontrols=0,controls=i+1)
    num_qubits_mod_add = 4*n+2
    targets_mod_add = np.arange(0,num_qubits_mod_add,1) + n + 1
    exponent = np.arange(0,n,1)
    result = 2 ** exponent
    temp = a * result
    temp = temp % N
    indices = np.arange(0,n,1)
    indices = indices*3+1
    indices = indices + n + 1
    for i in range(n-1,-1,-1):
        temp2 = temp[i] // result
        positions = temp2 & 1
        positions = positions == 1
        if(np.sum(positions)!=0):
            targets = indices[positions]
            gate.add_operation("X", targets=targets, controls=[0,i+1])
            rev_modadd = get_rev_modadd(N,n)
            gate.add_operation(rev_modadd, targets=targets_mod_add)
            gate.add_operation("X", targets=targets, controls=[0,i+1])
    return gate

In [ ]:
def get_mod_exp(a,N,n):
    num_qubits = 6*n+3
    gate = qs.QGate(num_qubits, 0, "EXPMOD") # Modular Exponentiation
    gate.add_operation("X", targets=n+1) # Initialize Register 1 to 1

    num_qubits_mod_mult = 5*n+3

    for i in range(n):
        #control qubit
        gate.add_operation("X", targets=n, controls=i)

        # MODMULT
        temp = a ** (2 ** i)
        temp = temp % N

        targets = np.arange(0,num_qubits_mod_mult,1)
        targets += n
        modmult = get_mod_mult(temp,N,n)
        gate.add_operation(modmult, targets=targets)

        # SWAP
        register_one_indices = np.arange(0,n,1)
        register_one_indices += n
        register_one_indices += 1

        register_b_indices = np.arange(0,n,1)
        register_b_indices *=3
        register_b_indices += 2
        register_b_indices += 2*n
        register_b_indices += 1

        for j in range(n):
            from_ = register_one_indices[j]
            to = register_b_indices[j]
            gate.add_operation("SWAP", targets=[from_,to])


        # Reverse MODMULT
        temp = a ** (2 ** i)
        temp = temp % N
        temp = invmodn(temp,N)
        rev_mod_mult = get_rev_mod_mult(temp,N,n)
        gate.add_operation(rev_mod_mult, targets=targets)

        # control qubit
        gate.add_operation("X", targets=n, controls=i)
    return gate

In [ ]:
a = 2
x = 7
N = 7
n = 3

size = 6*n+3
register_2 = n
register_1_plus_control = n + 1
adder_bits = 3*n+1

# modulo
modulo_bin= to_bin(N,n)
indices_modulo = np.arange(0,n,1)
indices_modulo += register_2
indices_modulo += register_1_plus_control
indices_modulo += adder_bits
input = np.zeros(size)
input[indices_modulo] = modulo_bin

# exponent
exponent = x
indices_exponent = np.arange(0,n,1)
exponent_bin= to_bin(exponent,n)
input[indices_exponent] = exponent_bin

c = qs.QCircuit(size, size, "EXPMOD", ancilla=input)
mod_exp = get_mod_exp(a,N,n)
targets = np.arange(0,size,1)
c.add_operation(mod_exp,targets=targets)
c.add_operation("MEASURE", targets=targets, outputs=targets)
use_system = False
executor = qs.Drewom(qmachine="doki",
                        extra={"num_threads": -1,
                                "random_generator": np.random.rand,
                                "use_system": use_system})
circuit = c
iterations = 1
mes = executor.execute(circuit, iterations=iterations)

    #print("RawMes:", mes)
indices = np.arange(0,n,1)
indices += n
indices += 1
mes = np.array(mes[0])
print(mes[indices]+0)
print(mes[indices_modulo]+0)
print(mes[indices_exponent]+0)

[0 1 0]
[1 1 1]
[1 1 1]


In [ ]:
class TestStringMethods(unittest.TestCase):

    def test_upper(self):
        self.assertEqual('foo'.upper(), 'FOO')

    def test_isupper(self):
        self.assertTrue('FOO'.isupper())
        self.assertFalse('Foo'.isupper())

    def test_split(self):
        s = 'hello world'
        self.assertEqual(s.split(), ['hello', 'world'])
        # check that s.split fails when the separator is not a string
        with self.assertRaises(TypeError):
            s.split(2)


In [ ]:
class TestModularExponentiation(unittest.TestCase):

    def test_adder(self):
        # arrange
        n = 4
        size = 3*n+1
        for i in range(2**n):
          for j in range(2**n):
            a = i
            b = j
            expected = a + b

            if expected > (2**n)-1:
              continue

            indices_register_a = np.arange(0,n,1)
            indices_register_a *= 3
            indices_register_a += 1
            a_bin = to_bin(a,n)

            indices_register_b = np.arange(0,n,1)
            indices_register_b *= 3
            indices_register_b += 2
            b_bin = to_bin(b,n)

            input = np.zeros(size)

            input[indices_register_a] = a_bin
            input[indices_register_b] = b_bin

            c = qs.QCircuit(size, size, "ADDER", ancilla=input)

            adder = get_adder(n)
            targets = np.arange(0,size,1)
            c.add_operation(adder, targets=targets)
            targets = np.arange(0,size,1)
            c.add_operation("MEASURE", targets=targets, outputs=targets)
            use_system = False
            executor = qs.Drewom(qmachine="doki",
                                extra={"num_threads": -1,
                                        "random_generator": np.random.rand,
                                        "use_system": use_system})
            circuit = c
            iterations = 1

            # act
            mes = executor.execute(circuit, iterations=iterations)

            mes = np.array(mes[0])
            temp = mes[indices_register_b]
            weights = np.arange(0,n,1)
            weights = 2 ** weights
            temp = temp * weights
            actual = np.sum(temp)

            # assert
            self.assertEqual(actual,expected)

    def test_ra(self):
        # arrange
        n = 4
        size = 3*n+1
        for i in range(2**n):
          for j in range(2**n):
            a = i
            b = j
            expected = b

            if expected > (2**n)-1:
              continue

            indices_register_a = np.arange(0,n,1)
            indices_register_a *= 3
            indices_register_a += 1
            a_bin = to_bin(a,n)

            indices_register_b = np.arange(0,n,1)
            indices_register_b *= 3
            indices_register_b += 2
            b_bin = to_bin(b,n)

            input = np.zeros(size)

            input[indices_register_a] = a_bin
            input[indices_register_b] = b_bin

            c = qs.QCircuit(size, size, "ADDER", ancilla=input)

            adder = get_adder(n)
            ra = get_ra(n)
            targets = np.arange(0,size,1)
            c.add_operation(adder, targets=targets)
            c.add_operation(ra,targets=targets)
            c.add_operation("MEASURE", targets=targets, outputs=targets)
            use_system = False
            executor = qs.Drewom(qmachine="doki",
                                extra={"num_threads": -1,
                                        "random_generator": np.random.rand,
                                        "use_system": use_system})
            circuit = c
            iterations = 1

            # act
            mes = executor.execute(circuit, iterations=iterations)
            mes = np.array(mes[0])
            temp = mes[indices_register_b]
            weights = np.arange(0,n,1)
            weights = 2 ** weights
            temp = temp * weights
            actual = np.sum(temp)

            # assert
            self.assertEqual(actual,expected)

    def test_modadd(self):
        N = 15
        n = 4
        size = 4*n+2
        modadd = get_modadd(N,n)
        # 0 <= a,b < N
        for i in range(N):
          for j in range(N):
            a = i
            b = j
            expected = (a + b) % N

            adder_bits = 3*n+1

            input = np.zeros(size)

            # modulo
            modulo_bin= to_bin(N,n)
            indices_modulo = np.arange(0,n,1)
            indices_modulo += adder_bits
            input[indices_modulo] = modulo_bin

            indices_register_a = np.arange(0,n,1)
            indices_register_a *= 3
            indices_register_a += 1
            a_bin = to_bin(a,n)

            indices_register_b = np.arange(0,n,1)
            indices_register_b *= 3
            indices_register_b += 2
            b_bin = to_bin(b,n)



            input[indices_register_a] = a_bin
            input[indices_register_b] = b_bin



            c = qs.QCircuit(size, size, "MODADD", ancilla=input)
            targets = np.arange(0,size,1)
            c.add_operation(modadd, targets=targets)
            c.add_operation("MEASURE", targets=targets, outputs=targets)
            use_system = False
            executor = qs.Drewom(qmachine="doki",
                                extra={"num_threads": -1,
                                        "random_generator": np.random.rand,
                                        "use_system": use_system})
            circuit = c
            iterations = 1
            # act
            mes = executor.execute(circuit, iterations=iterations)
            mes = np.array(mes[0])
            temp = mes[indices_register_b]
            weights = np.arange(0,n,1)
            weights = 2 ** weights
            temp = temp * weights
            actual = np.sum(temp)

            temp = mes[indices_register_a]
            temp = temp * weights
            register_a = np.sum(temp)

            temp = mes[indices_modulo]
            temp = temp * weights
            modulo = np.sum(temp)

            # assert
            #print(i,j)
            self.assertEqual(actual,expected)
            self.assertEqual(mes[size-1],0)
            self.assertEqual(register_a,a)
            self.assertEqual(modulo,N)

    def test_rev_modadd(self):
        N = 15
        n = 4
        size = 4*n+2
        modadd = get_modadd(N,n)
        rev_modadd = get_rev_modadd(N,n)
        # 0 <= a,b < N
        for i in range(N):
          for j in range(N):
            a = i
            b = j
            expected = b

            adder_bits = 3*n+1

            input = np.zeros(size)

            # modulo
            modulo_bin= to_bin(N,n)
            indices_modulo = np.arange(0,n,1)
            indices_modulo += adder_bits
            input[indices_modulo] = modulo_bin

            indices_register_a = np.arange(0,n,1)
            indices_register_a *= 3
            indices_register_a += 1
            a_bin = to_bin(a,n)

            indices_register_b = np.arange(0,n,1)
            indices_register_b *= 3
            indices_register_b += 2
            b_bin = to_bin(b,n)



            input[indices_register_a] = a_bin
            input[indices_register_b] = b_bin



            c = qs.QCircuit(size, size, "MODADD", ancilla=input)
            targets = np.arange(0,size,1)
            c.add_operation(modadd, targets=targets)
            c.add_operation(rev_modadd,targets=targets)
            c.add_operation("MEASURE", targets=targets, outputs=targets)
            use_system = False
            executor = qs.Drewom(qmachine="doki",
                                extra={"num_threads": -1,
                                        "random_generator": np.random.rand,
                                        "use_system": use_system})
            circuit = c
            iterations = 1
            # act
            mes = executor.execute(circuit, iterations=iterations)
            mes = np.array(mes[0])
            temp = mes[indices_register_b]
            weights = np.arange(0,n,1)
            weights = 2 ** weights
            temp = temp * weights
            actual = np.sum(temp)

            temp = mes[indices_register_a]
            temp = temp * weights
            register_a = np.sum(temp)

            temp = mes[indices_modulo]
            temp = temp * weights
            modulo = np.sum(temp)

            # assert
            #print(i,j)
            self.assertEqual(actual,expected)
            self.assertEqual(mes[size-1],0)
            self.assertEqual(register_a,a)
            self.assertEqual(modulo,N)

    def test_mod_mult_control_to_zero(self):
          a = 2
          x = 1
          N = 3
          n = 2

          expected = x

          size = 5*n+3
          register_1_plus_control = n + 1
          adder_bits = 3*n+1

          input = np.zeros(size)

          #control qubit
          input[0] = 0

          register_1 = to_bin(x,n)
          indices_register_1 = np.arange(0,n,1)
          indices_register_1 += 1
          input[indices_register_1] = register_1

          # modulo
          modulo_bin= to_bin(N,n)
          indices_modulo = np.arange(0,n,1)
          indices_modulo += adder_bits
          indices_modulo += register_1_plus_control
          input[indices_modulo] = modulo_bin


          indices_register_a = np.arange(0,n,1)
          indices_register_a *= 3
          indices_register_a += 1
          indices_register_a += register_1_plus_control

          indices_register_b = np.arange(0,n,1)
          indices_register_b *= 3
          indices_register_b += 2
          indices_register_b += register_1_plus_control

          c = qs.QCircuit(size, size, "MODMULT", ancilla=input)
          modmult = get_mod_mult(a,N,n)
          targets = np.arange(0,size,1)
          c.add_operation(modmult, targets=targets)
          targets = np.arange(0,size,1)
          c.add_operation("MEASURE", targets=targets, outputs=targets)
          use_system = False
          executor = qs.Drewom(qmachine="doki",
                              extra={"num_threads": -1,
                                      "random_generator": np.random.rand,
                                      "use_system": use_system})
          circuit = c
          iterations = 1
          mes = executor.execute(circuit, iterations=iterations)
          mes = np.array(mes[0])
          temp = mes[indices_register_b]
          weights = np.arange(0,n,1)
          weights = 2 ** weights
          temp = temp * weights
          actual = np.sum(temp)

          temp = mes[indices_register_a]
          temp = temp * weights
          register_a = np.sum(temp)

          temp = mes[indices_modulo]
          temp = temp * weights
          modulo = np.sum(temp)


          # assert
          #print(i,j)
          self.assertEqual(actual,expected)
          self.assertEqual(register_a,0)
          self.assertEqual(modulo,N)

    def test_mod_mult_control_to_one(self):
          a = 2
          x = 1
          N = 3
          n = 2

          expected = (a * x) % N

          size = 5*n+3
          register_1_plus_control = n + 1
          adder_bits = 3*n+1

          input = np.zeros(size)

          #control qubit
          input[0] = 1

          register_1 = to_bin(x,n)
          indices_register_1 = np.arange(0,n,1)
          indices_register_1 += 1
          input[indices_register_1] = register_1

          # modulo
          modulo_bin= to_bin(N,n)
          indices_modulo = np.arange(0,n,1)
          indices_modulo += adder_bits
          indices_modulo += register_1_plus_control
          input[indices_modulo] = modulo_bin


          indices_register_a = np.arange(0,n,1)
          indices_register_a *= 3
          indices_register_a += 1
          indices_register_a += register_1_plus_control

          indices_register_b = np.arange(0,n,1)
          indices_register_b *= 3
          indices_register_b += 2
          indices_register_b += register_1_plus_control

          c = qs.QCircuit(size, size, "MODMULT", ancilla=input)
          modmult = get_mod_mult(a,N,n)
          targets = np.arange(0,size,1)
          c.add_operation(modmult, targets=targets)
          targets = np.arange(0,size,1)
          c.add_operation("MEASURE", targets=targets, outputs=targets)
          use_system = False
          executor = qs.Drewom(qmachine="doki",
                              extra={"num_threads": -1,
                                      "random_generator": np.random.rand,
                                      "use_system": use_system})
          circuit = c
          iterations = 1
          mes = executor.execute(circuit, iterations=iterations)
          mes = np.array(mes[0])
          temp = mes[indices_register_b]
          weights = np.arange(0,n,1)
          weights = 2 ** weights
          temp = temp * weights
          actual = np.sum(temp)

          temp = mes[indices_register_a]
          temp = temp * weights
          register_a = np.sum(temp)

          temp = mes[indices_modulo]
          temp = temp * weights
          modulo = np.sum(temp)


          # assert
          #print(i,j)
          self.assertEqual(actual,expected)
          self.assertEqual(register_a,0)
          self.assertEqual(modulo,N)

    def test_rev_mod_mult_control_to_zero(self):
          a = 2
          x = 1
          N = 3
          n = 2

          expected = x

          size = 5*n+3
          register_1_plus_control = n + 1
          adder_bits = 3*n+1

          input = np.zeros(size)

          #control qubit
          input[0] = 0

          register_1 = to_bin(x,n)
          indices_register_1 = np.arange(0,n,1)
          indices_register_1 += 1
          input[indices_register_1] = register_1

          # modulo
          modulo_bin= to_bin(N,n)
          indices_modulo = np.arange(0,n,1)
          indices_modulo += adder_bits
          indices_modulo += register_1_plus_control
          input[indices_modulo] = modulo_bin


          indices_register_a = np.arange(0,n,1)
          indices_register_a *= 3
          indices_register_a += 1
          indices_register_a += register_1_plus_control

          indices_register_b = np.arange(0,n,1)
          indices_register_b *= 3
          indices_register_b += 2
          indices_register_b += register_1_plus_control

          c = qs.QCircuit(size, size, "MODMULT", ancilla=input)
          modmult = get_mod_mult(a,N,n)
          targets = np.arange(0,size,1)
          c.add_operation(modmult, targets=targets)
          rev_mod_mult = get_rev_mod_mult(invmodn(a,N),N,n)
          c.add_operation(rev_mod_mult,targets=targets)
          targets = np.arange(0,size,1)
          c.add_operation("MEASURE", targets=targets, outputs=targets)
          use_system = False
          executor = qs.Drewom(qmachine="doki",
                              extra={"num_threads": -1,
                                      "random_generator": np.random.rand,
                                      "use_system": use_system})
          circuit = c
          iterations = 1
          mes = executor.execute(circuit, iterations=iterations)
          mes = np.array(mes[0])
          temp = mes[indices_register_b]
          weights = np.arange(0,n,1)
          weights = 2 ** weights
          temp = temp * weights
          actual = np.sum(temp)

          temp = mes[indices_register_a]
          temp = temp * weights
          register_a = np.sum(temp)

          temp = mes[indices_modulo]
          temp = temp * weights
          modulo = np.sum(temp)


          # assert
          #print(i,j)
          self.assertEqual(actual,expected)
          self.assertEqual(register_a,0)
          self.assertEqual(modulo,N)

    def test_mod_exp(self):
        pass

In [ ]:
unittest.main(argv=[''], verbosity=2, exit=False)

test_adder (__main__.TestModularExponentiation.test_adder) ... ok
test_mod_exp (__main__.TestModularExponentiation.test_mod_exp) ... ok
test_mod_mult (__main__.TestModularExponentiation.test_mod_mult) ... ok
test_modadd (__main__.TestModularExponentiation.test_modadd) ... ok
test_ra (__main__.TestModularExponentiation.test_ra) ... ok
test_rev_mod_mult (__main__.TestModularExponentiation.test_rev_mod_mult) ... ok
test_rev_modadd (__main__.TestModularExponentiation.test_rev_modadd) ... ok
test_isupper (__main__.TestStringMethods.test_isupper) ... ok
test_split (__main__.TestStringMethods.test_split) ... ok
test_upper (__main__.TestStringMethods.test_upper) ... ok

----------------------------------------------------------------------
Ran 10 tests in 634.648s

OK
